# Sentiment Classification of Amazon Mobile Reviews with BERT

This notebook fine-tunes a BERT model for binary sentiment classification on Amazon mobile reviews and evaluates its performance on a held-out test set.

## 1. Environment setup

Installing dependencies and importing the libraries required for data preparation, model training, and evaluation.

In [ ]:
!pip install -q datasets evaluate transformers[sentencepiece] accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00


In [ ]:
from urllib.request import urlretrieve
import zipfile
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)

## 2. Loading the dataset

Downloading the Amazon mobile reviews dataset and loading it into a DataFrame.

In [ ]:
amazon_mobile_reviews_url =  "https://eduds.blob.core.windows.net/nlp/Amazon_Unlocked_Mobile.csv.zip"
filename = "Amazon_Unlocked_Mobile.csv.zip"

urlretrieve(amazon_mobile_reviews_url, filename)

with zipfile.ZipFile("/content/Amazon_Unlocked_Mobile.csv.zip") as zfile:
  zfile.extractall()

df = pd.read_csv("/content/Amazon_Unlocked_Mobile.csv")
df.head()

,Product Name,Brand Name,Price,Rating,Reviews,Review Votes
0,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,199.99,5,I feel so LUCKY to have found this used (phone...,1.0
1,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,199.99,4,"nice phone, nice up grade from my pantach revu...",0.0
2,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,199.99,5,Very pleased,0.0
3,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,199.99,4,It works good but it goes slow sometimes but i...,0.0
4,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,199.99,4,Great phone to replace my lost phone. The only...,0.0


## 3. Data filtering and label creation

Selecting review text and ratings, removing missing values, keeping only clearly positive and negative examples, and creating binary labels.

In [ ]:
df = df[["Rating", "Reviews"]]
df = df.dropna(subset=["Reviews"])
df = df[df["Rating"].isin([1, 5])]
df["label"] = df["Rating"].apply(lambda r: 1 if r == 5 else 0)

df.head(), df["label"].value_counts()


(    Rating                                            Reviews  label
 0        5  I feel so LUCKY to have found this used (phone...      1
 2        5                                       Very pleased      1
 5        1  I already had a phone with problems... I know ...      0
 8        5  I originally was using the Samsung S2 Galaxy f...      1
 11       5  This is a great product it came after two days...      1,
 label
 1    223575
 0     72335
 Name: count, dtype: int64)

In [ ]:
df = df.sample(10000, random_state=42)

## 4. Train-test split

Splitting the dataset into training and test sets and converting them to Hugging Face datasets.

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset,
})

dataset

DatasetDict({
    train: Dataset({
        features: ['Rating', 'Reviews', 'label'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['Rating', 'Reviews', 'label'],
        num_rows: 2000
    })
})

## 5. Tokenization and model initialization

Loading the tokenizer and BERT model, then tokenizing the review text for sequence classification.

In [ ]:
model_checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def tokenize_function(batch):
    return tokenizer(
        batch["Reviews"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["Reviews"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

tokenized_datasets

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Rating', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['Rating', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})

## 6. Evaluation metrics

Using accuracy and F1 score to evaluate classification performance.

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels)["f1"]
    return {"accuracy": acc, "f1": f1}

## 7. Training configuration

Defining training arguments and preparing the Trainer object.

In [ ]:
repo_name = "amazon-bert-sentiment"

training_args = TrainingArguments(
    run_name="amazon-bert-sentiment",
    output_dir=repo_name,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    push_to_hub=False,
    logging_steps=100,
    report_to="none",
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

## 8. Model training and evaluation

Fine-tuning the model on the training set and evaluating it on the test set.

In [ ]:
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.095614,0.123019,0.967000,0.978247
2,0.043614,0.139007,0.969500,0.979941


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

{'eval_loss': 0.12312782555818558,
 'eval_accuracy': 0.967,
 'eval_f1': 0.978246539222149,
 'eval_runtime': 13.5692,
 'eval_samples_per_second': 147.393,
 'eval_steps_per_second': 9.212,
 'epoch': 2.0}

## 9. Saving and publishing the model

Uploading the fine-tuned model and tokenizer to the Hugging Face Hub.

In [ ]:
trainer.push_to_hub()
tokenizer.push_to_hub(repo_name)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ntiment/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...ntiment/model.safetensors:  35%|###4      |  152MB /  438MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/alaszmigiel/amazon-bert-sentiment/commit/52c7c1c1dce95cf4eecb0926480e3e41e1ab8419', commit_message='Upload tokenizer', commit_description='', oid='52c7c1c1dce95cf4eecb0926480e3e41e1ab8419', pr_url=None, repo_url=RepoUrl('https://huggingface.co/alaszmigiel/amazon-bert-sentiment', endpoint='https://huggingface.co', repo_type='model', repo_id='alaszmigiel/amazon-bert-sentiment'), pr_revision=None, pr_num=None)

## 10. Inference example

Running a sample prediction with the fine-tuned model.

In [ ]:
model_id = "alaszmigiel/amazon-bert-sentiment"

clf = pipeline(
    "text-classification",
    model=model_id,
    tokenizer=model_id
)

clf("I really like this phone")

config.json:   0%|          | 0.00/815 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

[{'label': 'LABEL_1', 'score': 0.9987001419067383}]

## Conclusion

The fine-tuned BERT model can be used for binary sentiment classification of Amazon mobile reviews.  
The results show that transformer-based models are effective for this type of text classification task.